# **Dispatch / Combine 输入输出对应关系**

**导航**：[← 上一章：算子在 MoE 模型中的作用](02.04_dispatch_combine_operator.ipynb) | [返回课程主页](02.01_chapter_intro.ipynb) | [下一章：核心流程拆解 →](02.06_dispatch_combine_core_flow.ipynb)

---

本节聚焦大 EP（Expert-Parallel）MoE 通信算子最外层的接口契约：站在调用者视角，理清 `Dispatch` 与 `Combine` 各自的输入输出张量含义，以及 `Dispatch` 的输出如何被 `Combine` 复用。

**学习目标**：理解 Dispatch / Combine 的接口语义及二者之间的字段衔接关系。

**本章目录**（点击跳转）：
- [大 EP MoE 数据流总览](#1-大-ep-moe-数据流总览)
- [Dispatch / Combine 接口与衔接](#2-dispatch--combine-接口与衔接)
- [小结与自检](#3-小结与自检)

---

## **1. 大 EP MoE 数据流总览**

在 EP 并行下，每张卡承载一部分 MoE 专家，单次前向需要完成两次跨卡 all-to-all 通信：

![Combine](./images/02.05_overview_of_large_EP_MOE_data_flow.png)

图中虚线表示 `Dispatch` 在分发数据的同时还会产出一组**辅助信息**张量（`expandIdxOut`、`sendCountsOut`、`expertTokenNumsOut`），它们既用于辅助下游 FFN 计算，也是 `Combine` 回收数据时计算偏移、还原顺序所必需的依据。

---

## **2. Dispatch / Combine 接口与衔接**

### 2.1 Dispatch 接口

`Dispatch` 的核函数入参由 `Init` 接口定义：

```cpp
__aicore__ inline void Init(GM_ADDR shmemSpace, GM_ADDR x, GM_ADDR expertIds,
                            GM_ADDR expandXOut, GM_ADDR expandIdxOut, GM_ADDR expertTokenNumsOut,
                            GM_ADDR sendCountsOut,
                            GM_ADDR workspaceGM, TPipe *pipe, const DispatchTilingData *tilingData);
```

**输入**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">张量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">含义</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">形状 / 类型</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">来源</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>x</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本卡 batch 中的 token 特征</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>[bs, h]</code>，<code>XType = float16_t</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">上一层输出</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertIds</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每个 token 选中的 TopK 专家 id</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>[bs, k]</code>，<code>int32</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">路由模块（TopK 结果）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>shmemSpace</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win 区基址</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>GM_ADDR</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">通信库分配</td>
  </tr>
</table>

**输出**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">张量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">含义</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">类型</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expandXOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本卡 FFN 阶段需要处理的、从所有 rank 汇聚过来的 token 特征</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>ExpandXOutType = float16_t</code>（与输入同精度，非量化）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expandIdxOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每个被搬运 token 的三元组定位信息 <code>(rankId(本卡索引), tokenIndex(token索引), topkIdx(topk索引))</code>，按发送顺序写出</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int32</code>，每个 token 占 3 个元素</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertTokenNumsOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本卡上每个本地专家最终收到的 token 数（受 <code>expertTokenNumsType</code> 控制为 <code>cumsum(累加和)</code> 或 <code>count(非累加和)</code> 形式）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int64</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>sendCountsOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本卡发送给每对 <code>(对端 rank, 本地专家)</code> 的 token 计数表</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int32</code></td>
  </tr>
</table>

**Tiling 字段**（`DispatchTilingData`）：

```cpp
struct alignas(8) DispatchTilingData {
    uint32_t epWorldSize;          // EP 并行域中的 rank 总数
    uint32_t epRankId;             // 本卡的 rank id
    uint32_t moeExpertNum;         // MoE 全局专家总数
    uint32_t bs;                   // 本卡 batch size
    uint32_t k;                    // TopK 中的 K
    uint32_t h;                    // hidden size
    uint32_t aivNum;               // 使用的 AIV 核数
    uint32_t expertTokenNumsType;  // 0: cumsum 模式；1: count 模式
    uint32_t totalWinSizeEp;       // EP 域 Win 区总大小
};
```

### 2.2 Combine 接口

`Combine` 的核函数入参：

```cpp
__aicore__ inline void Init(GM_ADDR shmemSpace, GM_ADDR expandX, GM_ADDR expertIds, GM_ADDR expandIdx,
                            GM_ADDR epSendCount, GM_ADDR expertScales, GM_ADDR XOut, TPipe* pipe,
                            const MoeDistributeCombineShmemTilingData& tilingData);
```

**输入**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">张量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">含义</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">类型</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expandX</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本卡 FFN 计算后的 token 特征（待回送给源 rank）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>ExpandXType</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertIds</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">与 Dispatch 同一份 TopK 路由结果，用于按 token 回收 K 份 FFN 输出</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int32</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expandIdx</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">由 Dispatch 产出的 <code>(rankId, tokenIndex, topkIdx)</code> 三元组列表</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int32</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>epSendCount</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">由 Dispatch 产出的发送计数表（即 <code>sendCountsOut</code>），用于推算本核处理的 token 区间</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int32</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertScales</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">TopK 权重，用于对 K 份 FFN 结果加权求和</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>float32</code></td>
  </tr>
</table>

**输出**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">张量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">含义</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">类型</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>XOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">combine 后的本卡 token 特征</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>XType</code>（与 Dispatch 的 <code>x</code> 同精度）</td>
  </tr>
</table>

**Tiling 字段**（`MoeDistributeCombineShmemTilingData`）：

```cpp
struct MoeDistributeCombineShmemTilingData {
    uint32_t epWorldSize;
    uint32_t epRankId;
    uint32_t moeExpertNum;
    uint32_t moeExpertPerRankNum;  // 本卡承载的本地专家数 = moeExpertNum / epWorldSize
    uint32_t globalBs;             // = bs * epWorldSize
    uint32_t bs;
    uint32_t k;
    uint32_t h;
    uint32_t aivNum;
    uint64_t totalWinSize;
};
```

### 2.3 Dispatch → Combine 字段衔接

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">Dispatch 输出</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">Combine 输入</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">衔接作用</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expandXOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">（经 FFN 计算后）→ <code>expandX</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">FFN 处理后的 token，按相同顺序回送</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expandIdxOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">→ <code>expandIdx</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">提供「这一条 token 回去时应该写到哪个 rank、哪个 batch 位置、哪个 topk 槽位」</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>sendCountsOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">→ <code>epSendCount</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Combine 据此把待发送 token 等分到 AIV 核</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertTokenNumsOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">（在 FFN 阶段使用）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">告诉本卡每个本地专家收到了多少 token，FFN 内部需要</td>
  </tr>
</table>

**关键约束**：`Combine` 的 `expertIds` 必须与 `Dispatch` 输入的 `expertIds` 同源（同一份 TopK 结果），否则无法对齐 K 份 FFN 输出。

---

## **3. 小结与自检**

**小结**：Dispatch 把本卡 token 按 `expertIds` 跨卡发送，同时产出三组**辅助信息**张量（`expandIdxOut` / `sendCountsOut` / `expertTokenNumsOut`）；Combine 复用 `expandIdx` 与 `epSendCount` 完成反向 all-to-all，并按 `expertScales` 做 TopK 加权求和。

**自检题**：

1. 如果一个 token 被路由到 K 个不同的专家，Dispatch 会为它在 `expandIdxOut` 中写入几条三元组？
2. `sendCountsOut` 与 `expertTokenNumsOut` 都包含 token 数量信息，二者的统计维度有何不同？
3. 在 Combine 阶段，为什么必须同时拥有 `expandIdx` 与 `expertScales`？只用其中一个会缺失什么？

---

**导航**：[← 上一章：算子在 MoE 模型中的作用](02.04_dispatch_combine_operator.ipynb) | [返回课程主页](02.01_chapter_intro.ipynb) | [下一章：核心流程拆解 →](02.06_dispatch_combine_core_flow.ipynb)

---